Cell 1 – Imports and configuration

In [ ]:
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import datetime

# Inkscape config (for EMF export)
ENABLE_EMF_EXPORT = True          # Set False to disable EMF export
INKSCAPE_EXE = r"C:\Program Files\Inkscape\bin\inkscape.exe"   # Adjust to your system


rcParams['font.family'] = 'serif'
rcParams['font.size'] = 12
rcParams['axes.linewidth'] = 1.2
rcParams['lines.linewidth'] = 2.0
rcParams['figure.dpi'] = 150               

print("✅ Setup complete. Ready to go.\n")


Cell 2 – Function that computes the DTFT of the rectangular pulse

In [ ]:
def dtft_square_pulse(omega, N1):
    """
    Compute the DTFT of a discrete square pulse of length 2*N1+1.
    X(e^{j omega}) = sin(omega * (N1 + 0.5)) / sin(omega / 2)
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        numerator = np.sin(omega * (N1 + 0.5))
        denominator = np.sin(omega / 2)
        X = np.where(np.abs(denominator) < 1e-12, 2*N1+1, numerator / denominator)
    return X

print("✅ DTFT function defined.\n")

Cell 3 – Combined DTFT plot (all N₁ on one figure)

In [ ]:
# ==================================================
#  Choose your N1 values here
# ==================================================
N1_values = [1, 2, 3, 4, 5]   
# ==================================================


if not N1_values:
    raise ValueError("N1_values must contain at least one integer.")

print("⏳ Starting combined DTFT plot ...")

# Frequency axis: full range from -2π to 2π 
omega = np.linspace(-2*np.pi, 2*np.pi, 2000)

print("📈 Computing DTFT values for each N₁ ...")
fig, ax = plt.subplots(figsize=(10, 4.5))

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(N1_values)))
for N1, color in zip(N1_values, colors):
    X = dtft_square_pulse(omega, N1)
    ax.plot(omega, X, label=f'$N_1 = {N1}$', color=color, linewidth=1.5)

print("✏️ Adding annotations and styling ...")
N1_max = max(N1_values)
ax.annotate(
    f'${2*N1_max+1}$',
    xy=(0, 2*N1_max+1),
    xytext=(0.3, 2*N1_max+1 + 0.5),
    ha='left',
    fontsize=11,
    color='black'
)

ax.axhline(0, color='black', linewidth=0.6, linestyle=':')
ax.set_xlabel(r'$\omega$ (rad/sample)')
ax.set_ylabel(r'$X(e^{j\omega})$')
ax.set_title('DTFT of a discrete square pulse for different $N_1$')
ax.legend(loc='upper right', framealpha=0.7)

# x‑axis ticks: -2π, -π, 0, π, 2π
xticks = [-2*np.pi, -np.pi, 0, np.pi, 2*np.pi]
xticklabels = [r'$-2\pi$', r'$-\pi$', '0', r'$\pi$', r'$2\pi$']
ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)
ax.set_xlim(-2*np.pi, 2*np.pi)
ax.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()

# Export combined plot
export_dir = r"plots"  #### HERE U CAN TO ADJUST THE EXPORT DIRECTORY!! 
os.makedirs(export_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

combined_svg = os.path.join(export_dir, f"combined_dtft_{timestamp}.svg")
combined_png = os.path.join(export_dir, f"combined_dtft_{timestamp}.png")
combined_emf = os.path.join(export_dir, f"combined_dtft_{timestamp}.emf")

print("💾 Saving combined PNG ...")
fig.savefig(combined_png, dpi=300)
print("✔️  Saved PNG:", combined_png)

print("💾 Saving combined SVG ...")
fig.savefig(combined_svg)
print("✔️  Saved SVG:", combined_svg)

if ENABLE_EMF_EXPORT:
    print("🔄 Attempting combined EMF via Inkscape ...")
    try:
        if not os.path.isfile(INKSCAPE_EXE):
            raise FileNotFoundError(f"Inkscape not found at: {INKSCAPE_EXE}")
        cmd = [INKSCAPE_EXE, combined_svg, "--export-type=emf", f"--export-filename={combined_emf}"]
        subprocess.run(cmd, check=True)
        print("✔️  Saved EMF:", combined_emf)
    except Exception as e:
        print(f"[WARN] Combined EMF export failed: {e}")

plt.show()
print("✅ Combined plot finished.\n")

Cell 4 – Time‑domain rectangular pulse plot for each N₁

In [ ]:
print("⏳ Generating time‑domain rectangular pulse plots for each N₁ ...\n")

for N1 in N1_values:
    print(f"⏺ N₁ = {N1}")

    # Full time axis from -10 to 10 
    n_full = np.arange(-10, 11)             
    x_full = np.zeros_like(n_full, dtype=float)

    #   only ones inside the rectangular window [-N1, N1]
    x_full[(n_full >= -N1) & (n_full <= N1)] = 1

    fig_stem, ax_stem = plt.subplots(figsize=(8, 3.5))  
    markerline, stemlines, baseline = ax_stem.stem(n_full, x_full, basefmt=":")
    plt.setp(stemlines, linewidth=1.5, color='#1f77b4')
    plt.setp(markerline, markersize=6, color='#1f77b4')

    ax_stem.set_xlabel('n')
    ax_stem.set_ylabel('x[n]')
    ax_stem.set_title(f'Rectangular pulse ($N_1 = {N1}$)')

    
    ax_stem.set_xticks([-10, -N1, 0, N1, 10])
    ax_stem.set_xticklabels([r'$-10$', f'$-{N1}$', '0', f'${N1}$', r'$10$'])
    ax_stem.set_xlim(-10.5, 10.5)            
    ax_stem.set_yticks([0, 1])
    ax_stem.set_ylim(-0.2, 1.5)
    ax_stem.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    stem_svg = os.path.join(export_dir, f"rect_n1_{N1}_{ts}.svg")
    stem_png = os.path.join(export_dir, f"rect_n1_{N1}_{ts}.png")
    stem_emf = os.path.join(export_dir, f"rect_n1_{N1}_{ts}.emf")

    print("   💾 Saving PNG ...")
    fig_stem.savefig(stem_png, dpi=300)
    print("   ✔️  PNG:", stem_png)

    print("   💾 Saving SVG ...")
    fig_stem.savefig(stem_svg)
    print("   ✔️  SVG:", stem_svg)

    if ENABLE_EMF_EXPORT:
        print("   🔄 EMF via Inkscape ...")
        try:
            if not os.path.isfile(INKSCAPE_EXE):
                raise FileNotFoundError(f"Inkscape not found at: {INKSCAPE_EXE}")
            cmd = [INKSCAPE_EXE, stem_svg, "--export-type=emf", f"--export-filename={stem_emf}"]
            subprocess.run(cmd, check=True)
            print("   ✔️  EMF:", stem_emf)
        except Exception as e:
            print(f"   [WARN] EMF export failed: {e}")

    plt.show()
    print("")

print("✅ Time‑domain plots done.\n")

Cell 5 – Individual DTFT plot for each N₁

In [ ]:
print("⏳ Generating individual DTFT plots for each N₁ ...\n")

# Frequency range from -2π to 2π 
omega_ind = np.linspace(-2*np.pi, 2*np.pi, 2000)

for N1 in N1_values:
    print(f"⏺ N₁ = {N1}")
    X = dtft_square_pulse(omega_ind, N1)

    fig_ind, ax_ind = plt.subplots(figsize=(8, 4))
    ax_ind.plot(omega_ind, X, color='#d62728', linewidth=1.8, label=f'$N_1 = {N1}$')
    ax_ind.axhline(0, color='black', linewidth=0.6, linestyle=':')
    ax_ind.set_xlabel(r'$\omega$ (rad/sample)')
    ax_ind.set_ylabel(r'$X(e^{j\omega})$')
    ax_ind.set_title(f'DTFT of rectangular pulse ($N_1 = {N1}$)')
    ax_ind.legend(loc='upper right')

    
    ax_ind.set_xticks([-2*np.pi, -np.pi, 0, np.pi, 2*np.pi])
    ax_ind.set_xticklabels([r'$-2\pi$', r'$-\pi$', '0', r'$\pi$', r'$2\pi$'])
    ax_ind.set_xlim(-2*np.pi, 2*np.pi)
    ax_ind.grid(True, linestyle=':', alpha=0.6)

    
    ax_ind.annotate(
        f'{2*N1+1}',
        xy=(0, 2*N1+1),
        xytext=(0.3, 2*N1+1 + 0.5),
        fontsize=10,
        color='black'
    )

    plt.tight_layout()

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    ind_svg = os.path.join(export_dir, f"dtft_n1_{N1}_{ts}.svg")
    ind_png = os.path.join(export_dir, f"dtft_n1_{N1}_{ts}.png")
    ind_emf = os.path.join(export_dir, f"dtft_n1_{N1}_{ts}.emf")

    print("   💾 Saving PNG ...")
    fig_ind.savefig(ind_png, dpi=300)
    print("   ✔️  PNG:", ind_png)

    print("   💾 Saving SVG ...")
    fig_ind.savefig(ind_svg)
    print("   ✔️  SVG:", ind_svg)

    if ENABLE_EMF_EXPORT:
        print("   🔄 EMF via Inkscape ...")
        try:
            if not os.path.isfile(INKSCAPE_EXE):
                raise FileNotFoundError(f"Inkscape not found at: {INKSCAPE_EXE}")
            cmd = [INKSCAPE_EXE, ind_svg, "--export-type=emf", f"--export-filename={ind_emf}"]
            subprocess.run(cmd, check=True)
            print("   ✔️  EMF:", ind_emf)
        except Exception as e:
            print(f"   [WARN] EMF export failed: {e}")

    plt.show()
    print("")

print("✅ Individual DTFT plots done.\n")
print("🎉 All tasks completed. Check the plots/ folder for all files.")